# Analyze data
This script combines the resulting CSV-files from QuPath measurements and exports a combined CSV, as well as calculates some statistics from the data.

The expected filenames are ``<mousenum>_<cellID>.csv`` (e.g., ``196_1.csv``)

In [1]:
import pandas as pd
import numpy as np
from statistics import mean, stdev
import os
import re
import csv

In [2]:
# Specify data and output directories
data_directory = ('\\\\pn.vai.org\\projects\\moore\\primary\\vari-core-generated-data\\OIC\\AlexisBergsma\\20251118_PreliminaryAnalysis\\Measurements')

output_directory = ('\\\\pn.vai.org\\projects\\moore\\primary\\vari-core-generated-data\\OIC\\AlexisBergsma\\20251118_PreliminaryAnalysis\\Measurements')

In [3]:
# Find all CSV files in directory

contents = os.listdir(data_directory)

df_list = []
for c in contents:
    if os.path.isfile(os.path.join(data_directory, c)):
        if (c == 'combined.csv'  or c =='summary.csv'):
            continue
        elif c.endswith('.csv'):
            df = pd.read_csv(os.path.join(data_directory, c), header=0)
            
            if not df.empty:
                res = re.split(r'_|\.', c)        
                df['Mouse'] = int(res[0])
                df['Cell'] = int(res[1])
                df_list.append(df)

df_concat = pd.concat(df_list)
df_concat = df_concat.iloc[:, [3, 4, 0, 1, 2]]
df_concat = df_concat.sort_values(by=['Mouse', 'Cell', 'Image'])
df_concat.to_csv(os.path.join(output_directory, 'combined.csv'), index=False)


In [4]:
# Calculate statistics

print(list(df_concat)) # Print column headings

mouse_numbers = df_concat['Mouse'].unique()
cell_numbers = df_concat['Cell'].unique()
classifications = df_concat['Classification'].unique()

# print(mouse_numbers)
# print(cell_numbers)
# print(classifications)

with open(os.path.join(output_directory, 'summary.csv'), 'w', newline='') as csvfile:
    csvwriter = csv.writer(csvfile, delimiter=',')

    # Write headers
    csvwriter.writerow(['Mouse', 'Cell', 'Cell Area', 
                        'Num Mito', 'Mean Mito Area', 'StDev Mito Area', 'Mito Cell Fraction',
                        'Num Golgi', 'Mean Golgi Area', 'StDev Golgi Area', 'Golgi Cell Fraction'])

    for m in mouse_numbers:        

        mito_fractions = []
        golgi_fractions = []
        
        for ii in cell_numbers:

            # Select the rows that match this mouse and cell
            df_curr_cell = df_concat[(df_concat['Mouse'] == m) & 
                (df_concat['Cell'] == ii)]

            # Get cell area    
            curr_cell_area = df_curr_cell[df_curr_cell['Classification'] == 'Cell']['Area µm^2']
            curr_cell_area = curr_cell_area.iloc[0]
    
            df_curr_cell_mito = df_curr_cell[df_curr_cell['Classification'] == 'Mitochondria']
            num_mito = len(df_curr_cell_mito)
            mean_mito_area = df_curr_cell_mito['Area µm^2'].mean()

            stdev_mito_area = df_curr_cell_mito['Area µm^2'].std()
    
            mean_mito_to_cell = df_curr_cell_mito['Area µm^2'].sum() / curr_cell_area

            mito_fractions.append(mean_mito_to_cell)
    
            df_curr_cell_golgi = df_curr_cell[df_curr_cell['Classification'] == 'Golgi']
            num_golgi = len(df_curr_cell_golgi)
            mean_golgi_area = df_curr_cell_golgi['Area µm^2'].mean()
            stdev_golgi_area = df_curr_cell_golgi['Area µm^2'].std()
    
            mean_golgi_to_cell = (df_curr_cell_golgi['Area µm^2'].sum() / curr_cell_area)

            golgi_fractions.append(mean_golgi_to_cell)
    
            if ii == 1:
                csvwriter.writerow([m, ii, curr_cell_area, 
                                    num_mito, mean_mito_area, stdev_mito_area, mean_mito_to_cell,
                                    num_golgi, mean_golgi_area, stdev_golgi_area, mean_golgi_to_cell])
            else:
                csvwriter.writerow(['', ii, curr_cell_area, 
                                    num_mito, mean_mito_area, stdev_mito_area, mean_mito_to_cell,
                                    num_golgi, mean_golgi_area, stdev_golgi_area, mean_golgi_to_cell])

        # Calculate the average values for the whole mouse
        df_curr_mouse_mito = df_concat[(df_concat['Mouse'] == m) & 
        (df_concat['Classification'] == 'Mitochondria')]

        df_curr_mouse_golgi = df_concat[(df_concat['Mouse'] == m) & 
        (df_concat['Classification'] == 'Golgi')]

        df_curr_mouse_cell = df_concat[(df_concat['Mouse'] == m) & 
        (df_concat['Classification'] == 'Golgi')]
        
        csvwriter.writerow(['', 'Overall', df_curr_mouse_cell['Area µm^2'].mean(), 
                    len(df_curr_mouse_mito), df_curr_mouse_mito['Area µm^2'].mean(), df_curr_mouse_mito['Area µm^2'].std(), mean(mito_fractions),
                    len(df_curr_mouse_golgi), df_curr_mouse_golgi['Area µm^2'].mean(), df_curr_mouse_golgi['Area µm^2'].std(), mean(golgi_fractions)])

['Mouse', 'Cell', 'Image', 'Classification', 'Area µm^2']


In [5]:
# Sanity check

df_curr_mouse_mito = df_concat[(df_concat['Mouse'] == 194) & 
        (df_concat['Classification'] == 'Mitochondria')]

print(len(df_curr_mouse_mito))
print(df_curr_mouse_mito['Area µm^2'].mean())
print(df_curr_mouse_mito['Area µm^2'].std())

384
0.22161901041666665
0.1543726838282411


In [6]:
print(df_concat[df_concat['Mouse'] == 194])

    Mouse  Cell  Image Classification  Area µm^2
39    194     1  1.dm4           Cell   192.5900
0     194     1  3.dm4   Mitochondria     0.1303
1     194     1  3.dm4   Mitochondria     0.2439
2     194     1  3.dm4   Mitochondria     0.1391
3     194     1  3.dm4   Mitochondria     0.4699
..    ...   ...    ...            ...        ...
33    194    10  5.dm4   Mitochondria     0.0886
34    194    10  5.dm4   Mitochondria     0.0959
35    194    10  5.dm4   Mitochondria     0.3886
36    194    10  5.dm4   Mitochondria     0.0727
37    194    10  5.dm4   Mitochondria     0.1743

[439 rows x 5 columns]
